# Practice Lab: Multi-Tool Orchestration (Lesson 10.3)

Module 10 · 8 exercises · ReAct + loops + router + saga + budget + LangGraph + Reflexion

Exercises 1-5 run locally (pure Python).


# Lesson 10.3: Multi-Tool Orchestration

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python). Ex 6-8 are design/architecture.


In [ ]:
import json, time, hashlib
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): ReAct Agent Loop


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time

class ReActAgent:
    def __init__(self, max_steps=10, max_calls=5, timeout=30):
        self.max_steps = max_steps; self.max_calls = max_calls
        self.timeout = timeout; self.tools = {}
    def register(self, name, func): self.tools[name] = func
    def think(self, query, history):
        plan = []
        if "weather" in query.lower(): plan.append(("get_weather",{"city":"Hyderabad"}))
        if "cost" in query.lower() or "price" in query.lower():
            plan.append(("search_course",{"topic":"genai"}))
            plan.append(("calculate",{"expression":"14999*1.18"}))
        step = len(history)
        if step < len(plan): return {"type":"action","tool":plan[step][0],"args":plan[step][1]}
        return {"type":"answer","text":f"Done after {len(history)} observations"}

    def run(self, query):
        start = time.time(); history = []; step = 0; calls = 0
        print(f"  ReAct: '{query}'")
        while step < self.max_steps:
            if time.time()-start > self.timeout:
                return {"status":"timeout","steps":step}
            d = self.think(query, history)
            if d["type"] == "answer":
                print(f"  Step {step+1}: ANSWER -> {d['text']}")
                return {"status":"complete","steps":step+1,"calls":calls}
            if calls >= self.max_calls:
                print(f"  BUDGET: max calls ({self.max_calls})")
                return {"status":"budget_exceeded","steps":step+1,"calls":calls}
            t = self.tools.get(d["tool"])
            if t:
                r = t(**d["args"]); calls += 1; history.append(r)
                print(f"  Step {step+1}: {d['tool']}({d['args']}) -> {r}")
            step += 1
        return {"status":"max_steps","steps":step}

agent = ReActAgent(max_steps=10, max_calls=5)
agent.register("get_weather", lambda city: {"temp":34,"city":city})
agent.register("search_course", lambda topic: {"name":"GenAI","price":14999})
agent.register("calculate", lambda expression: {"result":eval(expression,{"__builtins__":{}})})

print("ReAct Agent:")
r1 = agent.run("Weather and GenAI course cost with GST?")
print(f"Result: {r1}\n")

agent2 = ReActAgent(max_steps=10, max_calls=2)
agent2.register("get_weather", lambda city: {"temp":34})
agent2.register("search_course", lambda topic: {"price":14999})
agent2.register("calculate", lambda expression: {"result":0})
r2 = agent2.run("Weather and course cost with GST?")
print(f"Budget test: {r2}")


</details>


---
## Exercise 2 (Easy): Loop Detection


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib, json

class LoopDetector:
    def __init__(self, max_repeats=2, max_consec=3):
        self.seen = {}; self.max_r = max_repeats
        self.max_c = max_consec; self.last = None; self.consec = 0

    def check(self, tool, args):
        h = hashlib.sha256(json.dumps({"t":tool,"a":args},sort_keys=True).encode()).hexdigest()[:16]
        self.seen[h] = self.seen.get(h, 0) + 1
        if self.seen[h] > self.max_r:
            return False, f"LOOP: {tool} repeated {self.seen[h]}x (max {self.max_r})"
        if tool == self.last: self.consec += 1
        else: self.consec = 1; self.last = tool
        if self.consec > self.max_c:
            return False, f"LOOP: {tool} {self.consec}x consecutive (max {self.max_c})"
        return True, "OK"

print("Loop Detection:")
d = LoopDetector(max_repeats=2, max_consec=3)
print("\n  Normal execution:")
for t, a in [("weather",{"city":"HYD"}),("search",{"q":"genai"}),("calc",{"e":"1+1"})]:
    ok, r = d.check(t, a); print(f"    {t}({a}) -> {r}")

print("\n  Infinite loop:")
d2 = LoopDetector(max_repeats=2, max_consec=3)
for i in range(5):
    ok, r = d2.check("get_stock", {"symbol":"RELIANCE"})
    print(f"    Call {i+1}: [{'OK' if ok else 'BLOCKED'}] {r}")
    if not ok: print(f"    -> Return error to LLM, suggest alternative"); break

print("\n  Consecutive same-tool:")
d3 = LoopDetector(max_repeats=5, max_consec=3)
for i in range(5):
    ok, r = d3.check("search", {"q":f"query_{i}"})
    print(f"    search(q='query_{i}') [{'OK' if ok else 'BLOCKED'}] {r}")
    if not ok: break


</details>


---
## Exercise 3 (Medium): Router Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Specialist:
    def __init__(self, name, kws, tools):
        self.name = name; self.kws = kws; self.tools = tools
    def handle(self, q):
        for tn, fn in self.tools.items():
            if any(k in q.lower() for k in tn.split("_")):
                return {"specialist":self.name,"tool":tn,"result":fn(q)}
        return {"specialist":self.name,"tool":"default","result":f"handled"}

class Router:
    def __init__(self): self.specs = []
    def add(self, s): self.specs.append(s)
    def route(self, q):
        best = max(self.specs, key=lambda s: sum(1 for k in s.kws if k in q.lower()), default=self.specs[0])
        return best.handle(q)

router = Router()
router.add(Specialist("weather",["weather","temperature","rain","forecast"],
    {"get_weather":lambda q:{"temp":34},"get_forecast":lambda q:{"days":5}}))
router.add(Specialist("finance",["price","cost","payment","emi","gst","refund"],
    {"get_price":lambda q:{"amount":14999},"check_refund":lambda q:{"eligible":True}}))
router.add(Specialist("courses",["course","enroll","module","lesson","genai","python","certificate"],
    {"search_courses":lambda q:{"top":"GenAI"},"get_syllabus":lambda q:{"modules":14}}))

tests = [("Weather in Mumbai?","weather"),("GenAI course cost?","finance"),
         ("Python course modules?","courses"),("Refund policy?","finance"),
         ("Rain tomorrow?","weather"),("Get certificate?","courses")]

print("Router Agent:")
correct = 0
for q, exp in tests:
    r = router.route(q)
    ok = r["specialist"] == exp; correct += ok
    print(f"  [{'OK' if ok else 'XX'}] '{q[:30]}' -> {r['specialist']} (exp: {exp})")
print(f"\n  Accuracy: {correct}/{len(tests)} = {correct/len(tests):.0%}")


</details>


---
## Exercise 4 (Medium): Saga Pattern


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Saga:
    def __init__(self): self.steps = []; self.done = []; self.trace = []
    def add(self, name, fwd, comp, tier="CRITICAL"):
        self.steps.append({"name":name,"fwd":fwd,"comp":comp,"tier":tier})
    def execute(self, ctx):
        for i, s in enumerate(self.steps):
            try:
                r = s["fwd"](ctx); self.done.append(s); ctx[s["name"]] = r
                self.trace.append({"step":s["name"],"action":"forward","result":r})
                print(f"    [{i+1}] {s['name']}: OK -> {r}")
            except Exception as e:
                self.trace.append({"step":s["name"],"action":"FAILED","error":str(e)})
                print(f"    [{i+1}] {s['name']}: FAILED -> {e}")
                if s["tier"] == "OPTIONAL": print(f"        (skipping)"); continue
                if s["tier"] == "IMPORTANT": print(f"        (warning)"); continue
                print(f"\n  Compensating {len(self.done)} steps:")
                for c in reversed(self.done):
                    cr = c["comp"](ctx)
                    self.trace.append({"step":c["name"],"action":"compensate","result":cr})
                    print(f"    UNDO {c['name']}: {cr}")
                return {"status":"rolled_back","failed":s["name"]}
        return {"status":"complete","steps":len(self.done)}

saga = Saga()
saga.add("book_flight", lambda c: {"booking":"FL-123","price":6500},
         lambda c: {"cancelled":"FL-123","refunded":6500})
saga.add("book_hotel",
         lambda c: (_ for _ in ()).throw(RuntimeError("Hotel full")),
         lambda c: {"cancelled":"HT-456"})
saga.add("rent_car", lambda c: {"booking":"CR-789"},
         lambda c: {"cancelled":"CR-789"}, tier="OPTIONAL")

print("Saga Pattern:")
r = saga.execute({})
print(f"\n  Result: {r}")
print(f"  Trace: {len(saga.trace)} entries")
for t in saga.trace:
    print(f"    {t['action']:<12} {t['step']:<15} {t.get('result',t.get('error',''))}")


</details>


---
## Exercise 5 (Medium): Token Budget Manager


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class BudgetMgr:
    def __init__(self, max_tok=128000):
        self.max = max_tok; self.msgs = []; self.total = 0; self.compactions = []
    def add(self, role, tokens):
        self.msgs.append({"role":role,"tokens":tokens}); self.total += tokens
    def util(self): return self.total / self.max
    def check(self):
        u = self.util()
        if u >= 0.90: return self._compact(10, "aggressive")
        elif u >= 0.85: return self._compact(5, "moderate")
        elif u >= 0.70: return "WARNING: 70%"
        return None
    def _compact(self, divisor, label):
        if len(self.msgs) <= 5: return "Too few"
        old = self.msgs[:-3]; old_tok = sum(m["tokens"] for m in old)
        summary_tok = max(old_tok // divisor, 50)
        self.msgs = [{"role":"system","tokens":summary_tok}] + self.msgs[-3:]
        saved = old_tok - summary_tok; self.total -= saved
        self.compactions.append({"type":label,"saved":saved})
        return f"{label}: saved {saved:,} tokens"

mgr = BudgetMgr(128000)
mgr.add("system", 500); mgr.add("system", 2500)

print("Token Budget Manager:")
for turn in range(1, 31):
    mgr.add("user", 100); mgr.add("assistant", 200)
    mgr.add("tool", 300); mgr.add("assistant", 300)
    action = mgr.check()
    if action and "saved" in str(action):
        print(f"  Turn {turn:>2}: {mgr.util():.0%} ({mgr.total:>7,} tok) -> {action}")
    elif turn % 5 == 0:
        print(f"  Turn {turn:>2}: {mgr.util():.0%} ({mgr.total:>7,} tok)")

total_saved = sum(c["saved"] for c in mgr.compactions)
print(f"\n  Final: {mgr.total:,}/{mgr.max:,} ({mgr.util():.0%})")
print(f"  Compactions: {len(mgr.compactions)}, saved: {total_saved:,} tokens")


</details>


---
## Exercise 6 (Challenge): LangGraph Stateful Workflow
Design/architecture. See practice-lab-10_3.html.


In [ ]:
# Exercise 6: LangGraph Stateful Workflow
pass


---
## Exercise 7 (Challenge): Reflexion Agent
Design/architecture. See practice-lab-10_3.html.


In [ ]:
# Exercise 7: Reflexion Agent
pass


---
## Exercise 8 (Challenge): India Voice Agent Pipeline
Design/architecture. See practice-lab-10_3.html.


In [ ]:
# Exercise 8: India Voice Agent Pipeline
pass
